# Percolation-04-Sharpness-Lean — la sharpness finie, prouvée en Lean 4

**Navigation** : [← Série Probas](../../README.md) · [Le lake `percolation_lean`](percolation_lean/) · [Jumeau Python](Percolation-04-Sharpness-Python.ipynb) · Voir #19494

Ce notebook est le **compagnon exécutable** du module [`Sharpness`](percolation_lean/Percolation/Sharpness.lean)
du lake `percolation_lean` — la tranche 5 du socle fini (#19494, item 3 du plan de croissance).
Il formalise une **inégalité de sharpness finie** : la probabilité (uniforme, `p = 1/2`) que la
composante ouverte d'un sommet `v` reste confinée à `k` sommets est au plus
`Σ 2^(−|δA|)` sommé sur les candidats `A ∋ v` de cardinal `≤ k`. C'est la forme **finie et
paramétrique** dont l'inégalité de décroissance exponentielle de l'article de référence sur la
percolation surcritique (Diskin–Easo–Radhakrishnan–Sudakov–Tassion, arXiv:2603.03257, §3.0)
est la limite : un profil isopérimétrique qui croît avec `|A|` suffit à écraser les petites
composantes.

Le module ajoute aussi le **chaînon manquant** de la trilogie Harris–Kleitman du socle : la forme
**mixte** (un événement croissant et un événement décroissant sont anti-corrélés). Prérequis
naturel : [`Percolation-02-Lean`](Percolation-02-Lean.ipynb) (le vocabulaire du noyau fini).

**Horizon honnête** : le socle reste **uniforme** (`p = 1/2`, mesure de comptage) — le cas
Bernoulli-`p` général, la grille infinie et la décroissance exponentielle quantitative de
l'article restent hors périmètre ; le comptage ci-dessous en est l'analogue fini, décidable et
vérifié par le kernel.

***

## 1. Les événements de confinement (décroissants)

Le socle de 02-Lean établissait que « être connecté » est **croissant** (`connectedEvent_isUpperSet`).
La sharpness vit de l'autre bord : les événements de **confinement** sont **décroissants** —
retirer des arêtes ouvertes ne peut qu'aggraver le confinement. Le module `Sharpness` en définit deux :

- `closedEvent G A` — les configurations `ω` où `A` est ω-fermé (aucune arête ouverte ne quitte `A`) ;
- `smallClusterEvent G v k` — les configurations `ω` où la composante ouverte de `v` a au plus `k` sommets.

In [1]:
-- Toutes les importations de la session viennent en tête (convention du kernel).
import Percolation
open Percolation

-- Les deux événements de confinement du module Sharpness :
#check @closedEvent            -- A est ω-fermé : aucune sortie ouverte
#check @smallClusterEvent      -- la composante de v a au plus k sommets

-- Ce sont des événements DÉCROISSANTS (le pendant de connectedEvent_isUpperSet) :
#check @closedEvent_isLowerSet
#check @smallClusterEvent_isLowerSet

#print axioms smallClusterEvent_isLowerSet

-- Toutes les importations de la session viennent en tête (convention du kernel).
import Percolation
open Percolation

-- Les deux événements de confinement du module Sharpness :
#check @closedEvent            -- A est ω-fermé : aucune sortie ouverte
──────▶  @closedEvent : {V : Type u_1} → (G : SimpleGraph V) → [Fintype (Edge G)] → Finset V → Finset (Finset (Edge G))
#check @smallClusterEvent      -- la composante de v a au plus k sommets
──────▶  @smallClusterEvent : {V : Type u_1} →
  [Fintype V] → (G : SimpleGraph V) → [Fintype (Edge G)] → V → ℕ → Finset (Finset (Edge G))

-- Ce sont des événements DÉCROISSANTS (le pendant de connectedEvent_isUpperSet) :
#check @closedEvent_isLowerSet
──────▶  @closedEvent_isLowerSet : ∀ {V : Type u_1} [Fintype V] [DecidableEq V] (G : SimpleGraph V) [inst : Fintype (Edge G)]
  [DecidableEq (Edge G)] (A : Finset V), IsLowerSet ↑(closedEvent G A)
#check @smallClusterEvent_isLowerSet
──────▶  @smallClusterEvent_isLowerSet : ∀ {V : Type u_1} [inst : Fintype V] [DecidableEq V] (G : SimpleGraph V)
  [inst_2 : Fintype (Edge G)] [DecidableEq (Edge G)] (v : V) (k : ℕ), IsLowerSet ↑(smallClusterEvent G v k)

#print axioms smallClusterEvent_isLowerSet
──────▶  'Percolation.smallClusterEvent_isLowerSet' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 0

Raw input:
{"cmd": "-- Toutes les importations de la session viennent en t\u00eate (convention du kernel).\nimport Percolation\nopen Percolation\n\n-- Les deux \u00e9v\u00e9nements de confinement du module Sharpness :\n#check @closedEvent            -- A est \u03c9-ferm\u00e9 : aucune sortie ouverte\n#check @smallClusterEvent      -- la composante de v a au plus k sommets\n\n-- Ce sont des \u00e9v\u00e9nements D\u00c9CROISSANTS (le pendant de connectedEvent_isUpperSet) :\n#check @closedEvent_isLowerSet\n#check @smallClusterEvent_isLowerSet\n\n#print axioms smallClusterEvent_isLowerSet"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "@closedEvent : {V : Type u_1} → (G : SimpleGraph V) → [Fintype (Edge G)] → Finset V → Finset (Finset (Edge G))"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "@smallClusterEvent : {V : Type u_1} →\n  [Fintype V] → (G : SimpleGraph V) → [Fintype (Edge G)] → V → ℕ → Finset (Finset (Edge G))"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "@closedEvent_isLowerSet : ∀ {V : Type u_1} [Fintype V] [DecidableEq V] (G : SimpleGraph V) [inst : Fintype (Edge G)]\n  [DecidableEq (Edge G)] (A : Finset V), IsLowerSet ↑(closedEvent G A)"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "@smallClusterEvent_isLowerSet : ∀ {V : Type u_1} [inst : Fintype V] [DecidableEq V] (G : SimpleGraph V)\n  [inst_2 : Fintype (Edge G)] [DecidableEq (Edge G)] (v : V) (k : ℕ), IsLowerSet ↑(smallClusterEvent G v k)"},
  {"severity": "info",
   "pos": {"line": 13, "column": 0},
   "endPos": {"line": 13, "column": 6},
   "data":
   "'Percolation.smallClusterEvent_isLowerSet' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 0}

**Lecture.** Les deux signatures portent la monotonie en plein texte :
`smallClusterEvent_isLowerSet` dit que si `ω` confine la composante de `v` à `k` sommets, toute
sous-configuration la confine encore — la composante ne peut que **grossir** quand on ouvre des
arêtes (`connected_mono`, tranche 2 du socle). Le `#print axioms` confirme une preuve kernel
sans `sorryAx` : le triplet standard de Mathlib seul. Ces deux événements décroissants sont la
matière première de toute l'anti-corrélation qui suit.

***

## 2. Harris–Kleitman mixte : croissant × décroissant = anti-corrélés

Le socle disposait des deux formes **pures** : deux croissants corrèlent positivement
(`harris_kleitman_upper_upper`), deux décroissants aussi (`harris_kleitman_lower_lower`).
Le chaînon manquant — celui que la sharpness réclame — est la forme **mixte** :

`#(𝒜 ∩ ℬ) · 2 ^ |α| ≤ #𝒜 · #ℬ`   (𝒜 croissant, ℬ décroissant)

En mesure uniforme : `P(𝒜 ∩ ℬ) ≤ P(𝒜) · P(ℬ)` — être confiné **défavorise** les grandes
connexions. C'est l'intuition surcritique à l'état fini.

In [2]:
-- La trilogie complète du socle, maintenant fermée :
#check @harris_kleitman_upper_upper    -- croissant × croissant (tranche 1)
#check @harris_kleitman_lower_lower    -- décroissant × décroissant (tranche 1)
#check @harris_kleitman_upper_lower    -- croissant × décroissant (tranche 5, nouveau)

-- L'instance probabiliste : connexité (croissant) × confinement (décroissant) :
#check @card_inter_connected_closed_le

-- La base d'axiomes : attendu le triplet standard de Mathlib, AUCUN sorryAx.
#print axioms harris_kleitman_upper_lower
#print axioms card_inter_connected_closed_le

-- La trilogie complète du socle, maintenant fermée :
#check @harris_kleitman_upper_upper    -- croissant × croissant (tranche 1)
──────▶  @harris_kleitman_upper_upper : ∀ {α : Type u_1} [inst : Fintype α] [inst_1 : DecidableEq α] (𝒜 ℬ : Finset (Finset α)),
  IsUpperSet ↑𝒜 → IsUpperSet ↑ℬ → 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card
#check @harris_kleitman_lower_lower    -- décroissant × décroissant (tranche 1)
──────▶  @harris_kleitman_lower_lower : ∀ {α : Type u_1} [inst : Fintype α] [inst_1 : DecidableEq α] (𝒜 ℬ : Finset (Finset α)),
  IsLowerSet ↑𝒜 → IsLowerSet ↑ℬ → 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card
#check @harris_kleitman_upper_lower    -- croissant × décroissant (tranche 5, nouveau)
──────▶  @harris_kleitman_upper_lower : ∀ {α : Type u_1} [inst : Fintype α] [inst_1 : DecidableEq α] (𝒜 ℬ : Finset (Finset α)),
  IsUpperSet ↑𝒜 → IsLowerSet ↑ℬ → (𝒜 ∩ ℬ).card * 2 ^ Fintype.card α ≤ 𝒜.card * ℬ.card

-- L'instance probabiliste : connexité (croissant) × confinement (décroissant) :
#check @card_inter_connected_closed_le
──────▶  @card_inter_connected_closed_le : ∀ {V : Type u_1} [Fintype V] [DecidableEq V] (G : SimpleGraph V)
  [inst : Fintype (Edge G)] [inst_1 : DecidableEq (Edge G)] (u v : V) (A : Finset V),
  (connectedEvent G u v ∩ closedEvent G A).card * 2 ^ Fintype.card (Edge G) ≤
    (connectedEvent G u v).card * (closedEvent G A).card

-- La base d'axiomes : attendu le triplet standard de Mathlib, AUCUN sorryAx.
#print axioms harris_kleitman_upper_lower
──────▶  'Percolation.harris_kleitman_upper_lower' depends on axioms: [propext, Classical.choice, Quot.sound]
#print axioms card_inter_connected_closed_le
──────▶  'Percolation.card_inter_connected_closed_le' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 1

Raw input:
{"cmd": "-- La trilogie compl\u00e8te du socle, maintenant ferm\u00e9e :\n#check @harris_kleitman_upper_upper    -- croissant \u00d7 croissant (tranche 1)\n#check @harris_kleitman_lower_lower    -- d\u00e9croissant \u00d7 d\u00e9croissant (tranche 1)\n#check @harris_kleitman_upper_lower    -- croissant \u00d7 d\u00e9croissant (tranche 5, nouveau)\n\n-- L'instance probabiliste : connexit\u00e9 (croissant) \u00d7 confinement (d\u00e9croissant) :\n#check @card_inter_connected_closed_le\n\n-- La base d'axiomes : attendu le triplet standard de Mathlib, AUCUN sorryAx.\n#print axioms harris_kleitman_upper_lower\n#print axioms card_inter_connected_closed_le", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@harris_kleitman_upper_upper : ∀ {α : Type u_1} [inst : Fintype α] [inst_1 : DecidableEq α] (𝒜 ℬ : Finset (Finset α)),\n  IsUpperSet ↑𝒜 → IsUpperSet ↑ℬ → 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@harris_kleitman_lower_lower : ∀ {α : Type u_1} [inst : Fintype α] [inst_1 : DecidableEq α] (𝒜 ℬ : Finset (Finset α)),\n  IsLowerSet ↑𝒜 → IsLowerSet ↑ℬ → 𝒜.card * ℬ.card ≤ 2 ^ Fintype.card α * (𝒜 ∩ ℬ).card"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@harris_kleitman_upper_lower : ∀ {α : Type u_1} [inst : Fintype α] [inst_1 : DecidableEq α] (𝒜 ℬ : Finset (Finset α)),\n  IsUpperSet ↑𝒜 → IsLowerSet ↑ℬ → (𝒜 ∩ ℬ).card * 2 ^ Fintype.card α ≤ 𝒜.card * ℬ.card"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data":
   "@card_inter_connected_closed_le : ∀ {V : Type u_1} [Fintype V] [DecidableEq V] (G : SimpleGraph V)\n  [inst : Fintype (Edge G)] [inst_1 : DecidableEq (Edge G)] (u v : V) (A : Finset V),\n  (connectedEvent G u v ∩ closedEvent G A).card * 2 ^ Fintype.card (Edge G) ≤\n    (connectedEvent G u v).card * (closedEvent G A).card"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"li

**Lecture.** `harris_kleitman_upper_lower` se déduit de la forme « deux décroissants » par
passage au complémentaire de l'événement croissant (le complémentaire d'un croissant est
décroissant) — l'arithmétique se joue en `ℕ` tronqué (`Nat.sub_mul`, `Nat.mul_sub`).
L'instance `card_inter_connected_closed_le` compose automatiquement : « `u` et `v` reliés » ×
« `A` fermé » — chaque membre du produit est la cardinalité d'un événement du socle, et
l'inégalité dit que leur intersection est **sous-représentée** par rapport à l'indépendance.
C'est le versant qualitatif de la sharpness ; le comptage des sections 4-5 en est le versant
quantitatif.

***

## 3. La frontière pleine : l'objet ω-indépendant du comptage

La tranche 4 (`Boundary`) définissait `boundary G ω A` — les arêtes **ouvertes** traversant `A`,
un objet qui dépend de l'état du monde `ω`. Le comptage réclame l'objet **indépendant de ω` :
la **frontière pleine** `graphBoundary G A` — toutes les arêtes de `G` (ouvertes ou fermées)
qui quittent `A`. Le pont entre les deux : `boundary = ω ∩ graphBoundary`.

Le levier du comptage : si `A` est ω-fermé, **aucune** arête de la frontière pleine n'est
ouverte — `|δA|` coordonnées sont **forcées fermées**.

In [3]:
-- La frontière pleine et son lien à la frontière ouverte de la tranche 4 :
#check @graphBoundary
#check @boundary_eq_inter_graphBoundary     -- boundary G ω A = ω ∩ graphBoundary G A
#check @disjoint_graphBoundary_of_openEdgeClosed

-- Le profil de la frontière pleine, évalué nativement :
#eval Fintype.card (Edge C3)                                        -- 3 arêtes au triangle
#eval (graphBoundary C3 ({0} : Finset (Fin 3))).card                -- sortie du singleton de C₃
#eval (graphBoundary C3 ({0, 1} : Finset (Fin 3))).card             -- sortie d'une paire adjacente
#eval (graphBoundary C4 ({0} : Finset (Fin 4))).card                -- singleton du carré
#eval (graphBoundary C4 ({0, 2} : Finset (Fin 4))).card             -- paire d'opposés du carré

-- La frontière pleine et son lien à la frontière ouverte de la tranche 4 :
#check @graphBoundary
──────▶  @graphBoundary : {V : Type u_1} →
  [Fintype V] →
    [DecidableEq V] →
      (G : SimpleGraph V) →
        [Fintype (Edge G)] → [DecidableEq (Edge G)] → [DecidableRel G.Adj] → Finset V → Finset (Edge G)
#check @boundary_eq_inter_graphBoundary     -- boundary G ω A = ω ∩ graphBoundary G A
──────▶  @boundary_eq_inter_graphBoundary : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V] (G : SimpleGraph V)
  [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [inst_4 : DecidableRel G.Adj] (ω : Finset (Edge G))
  (A : Finset V), boundary G ω A = ω ∩ graphBoundary G A
#check @disjoint_graphBoundary_of_openEdgeClosed
──────▶  @disjoint_graphBoundary_of_openEdgeClosed : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V]
  (G : SimpleGraph V) [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [inst_4 : DecidableRel G.Adj]
  {ω : Finset (Edge G)} {A : Finset V}, openEdgeClosed G ω ↑A → Disjoint ω (graphBoundary G A)

-- Le profil de la frontière pleine, évalué nativement :
#eval Fintype.card (Edge C3)                                        -- 3 arêtes au triangle
─────▶  3
#eval (graphBoundary C3 ({0} : Finset (Fin 3))).card                -- sortie du singleton de C₃
─────▶  2
#eval (graphBoundary C3 ({0, 1} : Finset (Fin 3))).card             -- sortie d'une paire adjacente
─────▶  2
#eval (graphBoundary C4 ({0} : Finset (Fin 4))).card                -- singleton du carré
─────▶  2
#eval (graphBoundary C4 ({0, 2} : Finset (Fin 4))).card             -- paire d'opposés du carré
─────▶  4
--% env 2

Raw input:
{"cmd": "-- La fronti\u00e8re pleine et son lien \u00e0 la fronti\u00e8re ouverte de la tranche 4 :\n#check @graphBoundary\n#check @boundary_eq_inter_graphBoundary     -- boundary G \u03c9 A = \u03c9 \u2229 graphBoundary G A\n#check @disjoint_graphBoundary_of_openEdgeClosed\n\n-- Le profil de la fronti\u00e8re pleine, \u00e9valu\u00e9 nativement :\n#eval Fintype.card (Edge C3)                                        -- 3 ar\u00eates au triangle\n#eval (graphBoundary C3 ({0} : Finset (Fin 3))).card                -- sortie du singleton de C\u2083\n#eval (graphBoundary C3 ({0, 1} : Finset (Fin 3))).card             -- sortie d'une paire adjacente\n#eval (graphBoundary C4 ({0} : Finset (Fin 4))).card                -- singleton du carr\u00e9\n#eval (graphBoundary C4 ({0, 2} : Finset (Fin 4))).card             -- paire d'oppos\u00e9s du carr\u00e9", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@graphBoundary : {V : Type u_1} →\n  [Fintype V] →\n    [DecidableEq V] →\n      (G : SimpleGraph V) →\n        [Fintype (Edge G)] → [DecidableEq (Edge G)] → [DecidableRel G.Adj] → Finset V → Finset (Edge G)"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data":
   "@boundary_eq_inter_graphBoundary : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V] (G : SimpleGraph V)\n  [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [inst_4 : DecidableRel G.Adj] (ω : Finset (Edge G))\n  (A : Finset V), boundary G ω A = ω ∩ graphBoundary G A"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "@disjoint_graphBoundary_of_openEdgeClosed : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V]\n  (G : SimpleGraph V) [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [inst_4 : DecidableRel G.Adj]\n  {ω : Finset (Edge G)} {A : Finset V}, openEdgeClosed G ω ↑A → Disjoint ω (graphBoundary G A)"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "3"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "2"},
  {"severity": "info",
   

**Lecture.** La frontière pleine étend le profil isopérimétrique de la tranche 4 hors de `ω` :
sur `C₃` le singleton **et** la paire adjacente ont exactement `2` sorties (le triangle est
1-connexe partout) ; sur `C₄` le singleton garde `2` sorties mais la paire d'**opposés** `{0,2}`
en a `4` — les deux faces du carré, exactement comme pour la frontière ouverte. La géométrie
du graphe parle dans `graphBoundary` **sans connaître l'état du monde** ; c'est ce qui permet
de compter des configurations.

***

## 4. Le comptage du confinement : `2^(N − |δA|)`

Premier compte : l'événement « `A` est ω-fermé » vit dans le sous-cube où les `|δA|` arêtes de
la frontière pleine sont fixées **fermées**. Il a donc au plus `2^(N − |δA|)` configurations
(`N` = nombre total d'arêtes) — c'est `card_closedEvent_le`.

Deuxième geste : la composante ouverte de `v` est **elle-même** ω-fermée (tranche 3). Donc si
elle a au plus `k` sommets, elle est l'un des **candidats au confinement** —
`confineCandidates G v k`, les parties `A ∋ v` de cardinal `≤ k` — et `ω` confine `v` dans `A`.

In [4]:
-- Le comptage élémentaire : confinement => sous-cube à |δA| coordonnées forcées :
#check @card_closedEvent_le

-- Les candidats au confinement de v, et la couverture de l'événement « petite composante » :
#check @confineCandidates
#check @smallClusterEvent_subset_biUnion

-- Les candidats k = 1 sur les graphes tests : exactement le singleton {v} :
#check @confineCandidates_C3_one
#check @confineCandidates_C4_one
#eval (confineCandidates (0 : Fin 3) 1).card
#eval (confineCandidates (0 : Fin 4) 1).card

-- Le comptage élémentaire : confinement => sous-cube à |δA| coordonnées forcées :
#check @card_closedEvent_le
──────▶  @card_closedEvent_le : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V] (G : SimpleGraph V)
  [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [inst_4 : DecidableRel G.Adj] (A : Finset V),
  (closedEvent G A).card ≤ 2 ^ (Finset.univ.card - (graphBoundary G A).card)

-- Les candidats au confinement de v, et la couverture de l'événement « petite composante » :
#check @confineCandidates
──────▶  @confineCandidates : {V : Type u_1} → [Fintype V] → [DecidableEq V] → V → ℕ → Finset (Finset V)
#check @smallClusterEvent_subset_biUnion
──────▶  @smallClusterEvent_subset_biUnion : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V] (G : SimpleGraph V)
  [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [DecidableRel G.Adj] (v : V) (k : ℕ),
  smallClusterEvent G v k ⊆ (confineCandidates v k).biUnion (closedEvent G)

-- Les candidats k = 1 sur les graphes tests : exactement le singleton {v} :
#check @confineCandidates_C3_one
──────▶  confineCandidates_C3_one : (confineCandidates 0 1).card = 1
#check @confineCandidates_C4_one
──────▶  confineCandidates_C4_one : (confineCandidates 0 1).card = 1
#eval (confineCandidates (0 : Fin 3) 1).card
─────▶  1
#eval (confineCandidates (0 : Fin 4) 1).card
─────▶  1
--% env 3

Raw input:
{"cmd": "-- Le comptage \u00e9l\u00e9mentaire : confinement => sous-cube \u00e0 |\u03b4A| coordonn\u00e9es forc\u00e9es :\n#check @card_closedEvent_le\n\n-- Les candidats au confinement de v, et la couverture de l'\u00e9v\u00e9nement \u00ab petite composante \u00bb :\n#check @confineCandidates\n#check @smallClusterEvent_subset_biUnion\n\n-- Les candidats k = 1 sur les graphes tests : exactement le singleton {v} :\n#check @confineCandidates_C3_one\n#check @confineCandidates_C4_one\n#eval (confineCandidates (0 : Fin 3) 1).card\n#eval (confineCandidates (0 : Fin 4) 1).card", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@card_closedEvent_le : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V] (G : SimpleGraph V)\n  [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [inst_4 : DecidableRel G.Adj] (A : Finset V),\n  (closedEvent G A).card ≤ 2 ^ (Finset.univ.card - (graphBoundary G A).card)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "@confineCandidates : {V : Type u_1} → [Fintype V] → [DecidableEq V] → V → ℕ → Finset (Finset V)"},
  {"severity": "info",
   "pos": {"line": 6, "column": 0},
   "endPos": {"line": 6, "column": 6},
   "data":
   "@smallClusterEvent_subset_biUnion : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V] (G : SimpleGraph V)\n  [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [DecidableRel G.Adj] (v : V) (k : ℕ),\n  smallClusterEvent G v k ⊆ (confineCandidates v k).biUnion (closedEvent G)"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data": "confineCandidates_C3_one : (confineCandidates 0 1).card = 1"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data": "confineCandidates_C4_one : (confineCandidates 0 1).card = 1"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "1"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "1"}],
 "env": 3}

**Lecture.** `smallClusterEvent_subset_bUnion` est la **couverture par union** : l'événement
« composante de `v` de cardinal `≤ k` » est inclus dans la réunion des `closedEvent G A` pour
`A` candidat — le témoin étant la composante elle-même (`component_self` pour `v ∈ A`,
`component_closed` pour la fermeture). Le `#eval` confirme le cas `k = 1` : un seul candidat,
le singleton `{v}`. Le théorème général n'énumère rien : il couvre, et chaque candidat paiera
son facteur `2^(N − |δA|)` dans la somme.

***

## 5. L'inégalité de sharpness finie

Le théorème tête du module — la couverture + le comptage élémentaire, assemblés par
bornes d'union et de somme :

`#(smallClusterEvent G v k) ≤ Σ_{A ∈ confineCandidates G v k} 2^(N − |δA|)`

Après normalisation par `2^N` (la mesure uniforme) :
`P(|C(v)| ≤ k) ≤ Σ_{A ∋ v, |A| ≤ k} 2^(−|δA|)` — la forme finie de l'inégalité de l'article
surcritique (arXiv:2603.03257, §3.0).

In [5]:
-- Le théorème tête, en comptage de configurations :
#check @card_smallClusterEvent_le

-- La base d'axiomes : le triplet standard de Mathlib, AUCUN sorryAx.
#print axioms card_smallClusterEvent_le

-- Le théorème tête, en comptage de configurations :
#check @card_smallClusterEvent_le
──────▶  @card_smallClusterEvent_le : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V] (G : SimpleGraph V)
  [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [inst_4 : DecidableRel G.Adj] (v : V) (k : ℕ),
  (smallClusterEvent G v k).card ≤ ∑ A ∈ confineCandidates v k, 2 ^ (Finset.univ.card - (graphBoundary G A).card)

-- La base d'axiomes : le triplet standard de Mathlib, AUCUN sorryAx.
#print axioms card_smallClusterEvent_le
──────▶  'Percolation.card_smallClusterEvent_le' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 4

Raw input:
{"cmd": "-- Le th\u00e9or\u00e8me t\u00eate, en comptage de configurations :\n#check @card_smallClusterEvent_le\n\n-- La base d'axiomes : le triplet standard de Mathlib, AUCUN sorryAx.\n#print axioms card_smallClusterEvent_le", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "@card_smallClusterEvent_le : ∀ {V : Type u_1} [inst : Fintype V] [inst_1 : DecidableEq V] (G : SimpleGraph V)\n  [inst_2 : Fintype (Edge G)] [inst_3 : DecidableEq (Edge G)] [inst_4 : DecidableRel G.Adj] (v : V) (k : ℕ),\n  (smallClusterEvent G v k).card ≤ ∑ A ∈ confineCandidates v k, 2 ^ (Finset.univ.card - (graphBoundary G A).card)"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "'Percolation.card_smallClusterEvent_le' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 4}

**Lecture.** La lecture probabiliste divise par `2^N` : chaque candidat `A` contribue au plus
`2^(−|δA|)` — un facteur **exponentiellement petit dès que la frontière pleine de `A` croît**.
Toute borne isopérimétrique `|δA| ≥ f(|A|)` transforme la somme en `Σ_k 2^(−f(k))` : c'est
exactement le mécanisme que l'article surcritique pousse à la limite (décroissance
exponentielle des probabilités de petites composantes dans la phase surcritique, grille
infinie). Ici, le mécanisme est **fini, paramétrique et prouvé au kernel** — le jumeau Python
[`Percolation-04-Sharpness-Python`](Percolation-04-Sharpness-Python.ipynb) en mesure la
réalisation numérique sur des grilles croissantes ; le Lean établit la forme exacte dont ces
mesures sont l'ombre.

***

## 6. Témoins `C₃` / `C₄` : la borne évaluée par le kernel

Sur les graphes tests du socle, le théorème se **calcule** : `k = 1`, un seul candidat `{0}`,
frontière pleine `2` — la somme de sharpness vaut `2^(3−2) = 2` sur `C₃` et `2^(4−2) = 4`
sur `C₄`. Chaque valeur est un théorème `by decide` : le kernel énumère et vérifie.

In [6]:
-- Les ingrédients, en théorèmes calculés :
#check @graphBoundary_C3_singleton      -- |δ{0}| = 2 sur C₃
#check @graphBoundary_C4_singleton      -- |δ{0}| = 2 sur C₄
#check @sharpness_sum_C3_one            -- la somme vaut exactement 2 = 2^(3−2)
#check @sharpness_sum_C4_one            -- la somme vaut exactement 4 = 2^(4−2)

-- Les bornes de sharpness qui en découlent :
#check @sharpness_C3_one                -- ≤ 2 configurations sur 2^3 = 8
#check @sharpness_C4_one                -- ≤ 4 configurations sur 2^4 = 16

#print axioms sharpness_C3_one

-- Les ingrédients, en théorèmes calculés :
#check @graphBoundary_C3_singleton      -- |δ{0}| = 2 sur C₃
──────▶  graphBoundary_C3_singleton : (graphBoundary C3 {0}).card = 2
#check @graphBoundary_C4_singleton      -- |δ{0}| = 2 sur C₄
──────▶  graphBoundary_C4_singleton : (graphBoundary C4 {0}).card = 2
#check @sharpness_sum_C3_one            -- la somme vaut exactement 2 = 2^(3−2)
──────▶  sharpness_sum_C3_one : ∑ A ∈ confineCandidates 0 1, 2 ^ (Finset.univ.card - (graphBoundary C3 A).card) = 2
#check @sharpness_sum_C4_one            -- la somme vaut exactement 4 = 2^(4−2)
──────▶  sharpness_sum_C4_one : ∑ A ∈ confineCandidates 0 1, 2 ^ (Finset.univ.card - (graphBoundary C4 A).card) = 4

-- Les bornes de sharpness qui en découlent :
#check @sharpness_C3_one                -- ≤ 2 configurations sur 2^3 = 8
──────▶  sharpness_C3_one : (smallClusterEvent C3 0 1).card ≤ 2
#check @sharpness_C4_one                -- ≤ 4 configurations sur 2^4 = 16
──────▶  sharpness_C4_one : (smallClusterEvent C4 0 1).card ≤ 4

#print axioms sharpness_C3_one
──────▶  'Percolation.sharpness_C3_one' depends on axioms: [propext, Classical.choice, Quot.sound]
--% env 5

Raw input:
{"cmd": "-- Les ingr\u00e9dients, en th\u00e9or\u00e8mes calcul\u00e9s :\n#check @graphBoundary_C3_singleton      -- |\u03b4{0}| = 2 sur C\u2083\n#check @graphBoundary_C4_singleton      -- |\u03b4{0}| = 2 sur C\u2084\n#check @sharpness_sum_C3_one            -- la somme vaut exactement 2 = 2^(3\u22122)\n#check @sharpness_sum_C4_one            -- la somme vaut exactement 4 = 2^(4\u22122)\n\n-- Les bornes de sharpness qui en d\u00e9coulent :\n#check @sharpness_C3_one                -- \u2264 2 configurations sur 2^3 = 8\n#check @sharpness_C4_one                -- \u2264 4 configurations sur 2^4 = 16\n\n#print axioms sharpness_C3_one", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "graphBoundary_C3_singleton : (graphBoundary C3 {0}).card = 2"},
  {"severity": "info",
   "pos": {"line": 3, "column": 0},
   "endPos": {"line": 3, "column": 6},
   "data": "graphBoundary_C4_singleton : (graphBoundary C4 {0}).card = 2"},
  {"severity": "info",
   "pos": {"line": 4, "column": 0},
   "endPos": {"line": 4, "column": 6},
   "data":
   "sharpness_sum_C3_one : ∑ A ∈ confineCandidates 0 1, 2 ^ (Finset.univ.card - (graphBoundary C3 A).card) = 2"},
  {"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 6},
   "data":
   "sharpness_sum_C4_one : ∑ A ∈ confineCandidates 0 1, 2 ^ (Finset.univ.card - (graphBoundary C4 A).card) = 4"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data": "sharpness_C3_one : (smallClusterEvent C3 0 1).card ≤ 2"},
  {"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data": "sharpness_C4_one : (smallClusterEvent C4 0 1).card ≤ 4"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "'Percolation.sharpness_C3_one' depends on axioms: [propext, Classical.choice, Quot.sound]"}],
 "env": 5}

**Lecture.** `sharpness_C3_one` dit : parmi les `2^3 = 8` configurations d'ouverture du
triangle, au plus **2** laissent la composante de `0` à un seul sommet — une proportion
`≤ 2/8 = 1/4 = 2^(−2)`. Sur le carré : au plus **4** sur `2^4 = 16`, encore `1/4`. Et la borne
est **exacte** sur `C₃` : les configurations qui isolent `0` sont précisément celles qui
ferment les deux arêtes incidentes (`0—1` et `0—2`), laissant la troisième libre — il y en a
exactement 2. Le `2^(−|δA|)` n'est pas une estimation paresseuse : c'est le compte exact du
sous-cube des arêtes forcées.

***

## 7. Exercices

Les trois exercices suivent la convention du dépôt : l'énoncé en commentaire, la
solution à décommenter puis exécuter. Tant que la solution est commentée, la cellule
reste neutre (`example : True := trivial`) et le notebook s'exécute de bout en bout.

In [7]:
-- Exercice 1 : le profil de la frontière pleine sur la paire adjacente du triangle.
-- Quelle valeur attendez-vous pour δ{0,1} sur C₃, sachant graphBoundary_C3_singleton ?
-- Décommentez et exécutez, puis confrontez au #eval de la section 3 :
-- #eval (graphBoundary C3 ({0, 1} : Finset (Fin 3))).card
-- #check @graphBoundary_C3_singleton

-- Indice : sur le triangle, TOUTE partie propre non vide a exactement 2 sorties — la seule
-- arête restante coupe A du reste. La frontière pleine prolonge le profil de la tranche 4.

example : True := trivial    -- cellule neutre tant que la solution est commentée

-- Exercice 1 : le profil de la frontière pleine sur la paire adjacente du triangle.
-- Quelle valeur attendez-vous pour δ{0,1} sur C₃, sachant graphBoundary_C3_singleton ?
-- Décommentez et exécutez, puis confrontez au #eval de la section 3 :
-- #eval (graphBoundary C3 ({0, 1} : Finset (Fin 3))).card
-- #check @graphBoundary_C3_singleton

-- Indice : sur le triangle, TOUTE partie propre non vide a exactement 2 sorties — la seule
-- arête restante coupe A du reste. La frontière pleine prolonge le profil de la tranche 4.

example : True := trivial    -- cellule neutre tant que la solution est commentée
--% env 6

Raw input:
{"cmd": "-- Exercice 1 : le profil de la fronti\u00e8re pleine sur la paire adjacente du triangle.\n-- Quelle valeur attendez-vous pour \u03b4{0,1} sur C\u2083, sachant graphBoundary_C3_singleton ?\n-- D\u00e9commentez et ex\u00e9cutez, puis confrontez au #eval de la section 3 :\n-- #eval (graphBoundary C3 ({0, 1} : Finset (Fin 3))).card\n-- #check @graphBoundary_C3_singleton\n\n-- Indice : sur le triangle, TOUTE partie propre non vide a exactement 2 sorties \u2014 la seule\n-- ar\u00eate restante coupe A du reste. La fronti\u00e8re pleine prolonge le profil de la tranche 4.\n\nexample : True := trivial    -- cellule neutre tant que la solution est comment\u00e9e", "env": 5}
Raw output:
{"env": 6}

In [8]:
-- Exercice 2 : auditer l'anti-corrélation connexité × confinement.
-- Quelles sont les deux hypothèses de monotonie de harris_kleitman_upper_lower, et qui les
-- fournit dans card_inter_connected_closed_le ? Décommentez et exécutez :
-- #print axioms card_inter_connected_closed_le
-- #check @connectedEvent_isUpperSet
-- #check @closedEvent_isLowerSet

-- Indice : attendez-vous au triplet standard (propext, Classical.choice, Quot.sound) et à
-- AUCUN sorryAx. L'anti-corrélation est une preuve kernel qui ne dépend d'aucun comptage.

example : True := trivial    -- cellule neutre tant que la solution est commentée

-- Exercice 2 : auditer l'anti-corrélation connexité × confinement.
-- Quelles sont les deux hypothèses de monotonie de harris_kleitman_upper_lower, et qui les
-- fournit dans card_inter_connected_closed_le ? Décommentez et exécutez :
-- #print axioms card_inter_connected_closed_le
-- #check @connectedEvent_isUpperSet
-- #check @closedEvent_isLowerSet

-- Indice : attendez-vous au triplet standard (propext, Classical.choice, Quot.sound) et à
-- AUCUN sorryAx. L'anti-corrélation est une preuve kernel qui ne dépend d'aucun comptage.

example : True := trivial    -- cellule neutre tant que la solution est commentée
--% env 7

Raw input:
{"cmd": "-- Exercice 2 : auditer l'anti-corr\u00e9lation connexit\u00e9 \u00d7 confinement.\n-- Quelles sont les deux hypoth\u00e8ses de monotonie de harris_kleitman_upper_lower, et qui les\n-- fournit dans card_inter_connected_closed_le ? D\u00e9commentez et ex\u00e9cutez :\n-- #print axioms card_inter_connected_closed_le\n-- #check @connectedEvent_isUpperSet\n-- #check @closedEvent_isLowerSet\n\n-- Indice : attendez-vous au triplet standard (propext, Classical.choice, Quot.sound) et \u00e0\n-- AUCUN sorryAx. L'anti-corr\u00e9lation est une preuve kernel qui ne d\u00e9pend d'aucun comptage.\n\nexample : True := trivial    -- cellule neutre tant que la solution est comment\u00e9e", "env": 6}
Raw output:
{"env": 7}

In [9]:
-- Exercice 3 : la borne de sharpness est-elle exacte sur C₃ ?
-- Comptez à la main : combien de configurations laissent la composante de 0 à un seul
-- sommet ? Confrontez votre compte à la borne, puis décommentez :
-- #check @sharpness_C3_one
-- #check @graphBoundary_C3_singleton

-- Indice : isoler 0 = fermer 0—1 ET 0—2 ; l'arête 1—2 reste libre (2 choix). Donc 2
-- configurations sur 8, exactement la borne 2^(3−2) : le sous-cube des arêtes forcées
-- est TOUT le compte — la borne de l'union est serrée au candidat {0}.

example : True := trivial    -- cellule neutre tant que la solution est commentée

-- Exercice 3 : la borne de sharpness est-elle exacte sur C₃ ?
-- Comptez à la main : combien de configurations laissent la composante de 0 à un seul
-- sommet ? Confrontez votre compte à la borne, puis décommentez :
-- #check @sharpness_C3_one
-- #check @graphBoundary_C3_singleton

-- Indice : isoler 0 = fermer 0—1 ET 0—2 ; l'arête 1—2 reste libre (2 choix). Donc 2
-- configurations sur 8, exactement la borne 2^(3−2) : le sous-cube des arêtes forcées
-- est TOUT le compte — la borne de l'union est serrée au candidat {0}.

example : True := trivial    -- cellule neutre tant que la solution est commentée
--% env 8

Raw input:
{"cmd": "-- Exercice 3 : la borne de sharpness est-elle exacte sur C\u2083 ?\n-- Comptez \u00e0 la main : combien de configurations laissent la composante de 0 \u00e0 un seul\n-- sommet ? Confrontez votre compte \u00e0 la borne, puis d\u00e9commentez :\n-- #check @sharpness_C3_one\n-- #check @graphBoundary_C3_singleton\n\n-- Indice : isoler 0 = fermer 0\u20141 ET 0\u20142 ; l'ar\u00eate 1\u20142 reste libre (2 choix). Donc 2\n-- configurations sur 8, exactement la borne 2^(3\u22122) : le sous-cube des ar\u00eates forc\u00e9es\n-- est TOUT le compte \u2014 la borne de l'union est serr\u00e9e au candidat {0}.\n\nexample : True := trivial    -- cellule neutre tant que la solution est comment\u00e9e", "env": 7}
Raw output:
{"env": 8}

## Conclusion

Le module `Sharpness` ferme le socle fini : la trilogie Harris–Kleitman est **complète**
(croissant-croissant, décroissant-décroissant, et le mixte croissant-décroissant,
anti-corrélé), le comptage du confinement est **démontré** (`2^(N − |δA|)` par candidat), et
l'**inégalité de sharpness finie** est établie — chaque preuve auditée par `#print axioms`
(triplet standard de Mathlib seul, zéro `sorry`).

Ce que le module **ne** prouve pas (encore) : le passage à la mesure Bernoulli-`p` générale,
la grille infinie, et la décroissance exponentielle quantitative de l'article surcritique —
autant de limites qui exigent des outils d'analyse hors d'un socle décidable. Le comptage
fini en est l'ossature paramétrique : le jumeau Python
[`Percolation-04-Sharpness-Python`](Percolation-04-Sharpness-Python.ipynb) en montre la
réalisation numérique ; le tracking général vit dans #14871.

Pour les preuves elles-mêmes : [`percolation_lean/Percolation/`](percolation_lean/Percolation/)
— six modules FR et leurs miroirs EN (convention i18n #4980), build `lake build` sans `sorry`.